# 04. Scenario 3 — "I want to verify this overseas approval alert" (risk verification sub-flow)

The third support scenario. The tool succeeds, but its result carries a **fraud/AML monitoring flag**. When D7 (risk check) detects it, the case passes through the conditional sub-flow **run the KYC re-verification tool -> D8 (pass/fail decision)** before closing.

In the second half you will experience **guardrail over-blocking** on both backends — the same situation phrased as a victim report is handled normally by Haiku, but blocked at intake by SGLang (Qwen3-VL-8B).


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.SCENARIOS[3]
print("Scenario:", sc["title_en"])
print("Customer question (Korean input):", sc["question"])
print("English gloss:", sc["question_en"])
print("script (pre-set for reproducibility):", sc["script"])   # risk_flag: True, kyc_result: verified


## Where this scenario sits in the full workflow

The path this scenario takes (red borders): the successful tool result carries a monitoring flag, so the case passes through the **risk_check -> kyc_verify (D8)** sub-flow before closing. This is the financial-industry rule "re-verify identity before closing a high-risk transaction", expressed in the graph.

Assigned team (red box): **card_security (lost/stolen/security)** — a suspicious-charge verification context. The model sometimes picks cards instead; compare with the D2 decision in the STEP output.
> Note: this diagram is not auto-generated from the running LangGraph; it is a **visualization-only drawing** rendered from the caption dictionaries in `ws_common.py`, keyed by node name (details: `[EN]_01_build_agent.ipynb`).


In [ ]:
W.draw_graph(highlight=W.SCENARIO_PATHS[3], show_teams=True, width=760)

## (1) Run with Haiku 4.5 decisions

In the STEP output, confirm that after risk_check decides `needs_verification`, the **kyc_verify node runs as an extra step**.


In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## (2) Run with SGLang (Qwen3-VL-8B) decisions


In [ ]:
res_sglang = W.run_verbose(graphs["SGLang"], sc["question"], sc["script"], "SGLang")

## (3) Execution time comparison


In [ ]:
W.compare(res_haiku, res_sglang)

## (4) Teaching moment: guardrail over-blocking — experience it on both backends

The scenario question above was deliberately written in neutral wording. Now we rephrase the same situation as a **victim report** — the way a customer would actually say it (Korean: "제가 쓰지 않은 해외 결제 승인 알림이 왔어요... 막아주세요" = "I got an approval alert for an overseas charge I did not make... please block it") — and run it through both arms unchanged.

- **Haiku arm**: D1 decides in_scope -> the full workflow proceeds normally (all the way to KYC).
- **SGLang arm**: D1 **over-blocks** with sensitive_compliance -> the run ends after 2 STEPs with a refusal reply. Read the final reply to see what a legitimate customer being refused looks like.


In [ ]:
victim_q = ("방금 제가 쓰지 않은 해외 결제 승인 알림이 왔어요. LUXE GADGETS라는 곳에서 "
            "USD 1,284이 결제됐다고 합니다. 카드 끝자리는 7712예요. 막아주세요.")

res_haiku_v = W.run_verbose(graphs["Haiku"], victim_q, sc["script"], "Haiku (victim phrasing)")

In [ ]:
res_sglang_v = W.run_verbose(graphs["SGLang"], victim_q, sc["script"], "SGLang (victim phrasing)")

### Compare D1 alone across phrasings

Three phrasings go to D1 only, with both backends side by side. Note that on the phrasings SGLang over-blocks, its **confidence saturates at 1.0** — a type of error that no threshold can filter.


In [ ]:
from IPython.display import Markdown, display
from agent.prompts import D1_QUESTION
from agent.state import D1_CHOICES
from agent.decisions import SGLangDecision, HaikuDecision
from common import BEDROCK_MODEL_IDS, PRICING

sg = SGLangDecision(model="Qwen/Qwen3-VL-8B-Instruct")
hk = HaikuDecision(client, BEDROCK_MODEL_IDS["haiku-4-5"], PRICING["haiku-4-5"])
probes = [
    victim_q,
    "오늘 새벽에 해외 결제 승인 알림이 왔는데 기억에 없는 거래예요. 확인해 주세요.",
    sc["question"],   # the neutral phrasing used by this scenario
]
rows = ["| phrase (Korean) | SGLang D1 (confidence) | Haiku D1 |", "|---|---|---|"]
for t in probes:
    r_sg = sg.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    r_hk = hk.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    rows.append(f"| {t[:38]}... | {r_sg.label} ({r_sg.confidence:.3f}) | {r_hk.label} |")
display(Markdown("\n".join(rows)))

**What to look for**
- Same customer, same situation — yet **depending on the phrasing**, the SGLang arm refuses at intake. This is the classic limitation of a small decision model misreading Korean victim-report language as a compliance violation.
- Because the confidence saturates at 1.0, a cascade of "escalate to a bigger model when confidence is low" cannot catch it. The realistic production setup is therefore a **hybrid: keep the safety decision (D1) on the larger model, move routing/review decisions (D2-D8) to the decision layer**.
- Even in the hybrid, only D1 is a Bedrock call, so most of the decision-latency gain is preserved.
- Next: `[EN]_05_scenario4_image.ipynb` — support cases where the model reads screenshots directly.
